# Portfolio Recommender Using ML Outputs

Goal: combine predicted alpha, existing quality scores, and current regime/risk signals into prototype conservative, balanced, and aggressive portfolios.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import numpy as np
import pandas as pd

from ai_models.feature_builder import build_feature_table
from ai_models.quality_score_model import run_quality_score_model
from ai_models.regime_detection_model import run_regime_detection_model
from ai_models.risk_detector import run_systemic_risk_detector

features = build_feature_table(
    fundamentals_path="data/fundamentals_cache.parquet",
    prices_path="data/prices_cache.parquet",
    treasury_path="data/treasury_yields_cache.parquet",
    benchmark_ticker="SPY",
).features.reset_index()
quality = run_quality_score_model(features)
regime = run_regime_detection_model(prices_path="data/prices_cache.parquet", treasury_path="data/treasury_yields_cache.parquet")
risk = run_systemic_risk_detector(prices_path="data/prices_cache.parquet", treasury_path="data/treasury_yields_cache.parquet")

In [3]:
# Placeholder ML output until a learned model is added.
candidate = features.merge(quality, on="Ticker", how="left")
candidate["PredictedAlpha"] = (
    candidate["QualityScore"].fillna(0.0) / 100.0
    + candidate["Momentum_12M"].fillna(0.0) * 0.5
    - candidate["Volatility_63D"].fillna(0.0) * 0.2
)
candidate[["Ticker", "QualityScore", "Momentum_12M", "Volatility_63D", "PredictedAlpha"]].head()

,Ticker,QualityScore,Momentum_12M,Volatility_63D,PredictedAlpha
0,A,53.558336,-0.094352,0.256801,0.437047
1,AAPL,68.177801,0.170154,0.236271,0.719601
2,ABBV,72.108711,-0.010978,0.271282,0.661342
3,ABNB,59.726650,0.038696,0.332178,0.550179
4,ABT,49.791031,-0.131591,0.272315,0.377652


In [4]:
latest_regime = regime.iloc[-1].to_dict() if not regime.empty else {}
latest_risk = risk.iloc[-1].to_dict() if not risk.empty else {}
latest_regime, latest_risk

({'Date': Timestamp('2026-03-19 00:00:00'),
  'RegimeLabel': 'Neutral',
  'ConfidenceScore': 0.55},
 {'Date': Timestamp('2026-03-19 00:00:00'),
  'RiskScore': 30.596162054362765,
  'RiskLevel': 'Low',
  'RiskFlags': 'Correlation spike',
  'Explanation': 'Primary drivers: cross-asset correlation, drawdown acceleration'})

In [5]:
conservative = candidate.sort_values(["QualityScore", "Volatility_63D"], ascending=[False, True]).head(10)
balanced = candidate.sort_values(["PredictedAlpha", "QualityScore"], ascending=[False, False]).head(10)
aggressive = candidate.sort_values(["PredictedAlpha", "Momentum_12M"], ascending=[False, False]).head(10)

conservative[["Ticker", "PredictedAlpha", "QualityScore", "Volatility_63D"]]

,Ticker,PredictedAlpha,QualityScore,Volatility_63D
301,MA,0.692318,77.861883,0.245567
352,NVDA,0.978458,77.595897,0.350602
428,SPG,0.837286,76.770328,0.185117
483,V,0.652189,74.968433,0.225438
504,WMB,0.858579,74.374278,0.204677
290,LLY,0.711932,73.323992,0.404884
218,GOOGL,1.141487,73.209452,0.219305
217,GOOG,1.123647,73.017513,0.217591
457,TPL,0.727731,72.950368,0.475023
29,AMGN,0.726987,72.445617,0.304565
